# 05 — Behavioral Classifier Bias Audit

**Mandatory per Phase 2 spec and ethics charter.**

Checks:
1. Demographic parity — flag rate deviation > 5pp across groups
2. Equalized odds — TPR (recall) across groups
3. False positive rate parity — FPR across groups
4. Cultural normality review — SHAP magnitude by group
5. Mitigation if needed — threshold adjustment, re-weighting, or feature exclusion

Finding and correcting bias is stronger evidence than presenting no audit.

In [1]:
import sys
from pathlib import Path

repo_root = Path(".").resolve().parent
sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit

from src.risk_classifier import FEATURE_COLS, DEMOGRAPHIC_COLS, TARGET_COL, load_model, score_to_band

df = pd.read_csv(repo_root / "data" / "synthetic" / "student_wellbeing.csv")
bundle = load_model()

# Use full dataset for demographic analysis (larger sample = more reliable group estimates)
X = df[FEATURE_COLS].values
y = df[TARGET_COL].values
probs = bundle.calibrated_pipeline.predict_proba(X)[:, 1]
preds = (probs >= bundle.threshold).astype(int)
df["pred_score"] = probs * 100
df["pred_label"] = preds

print(f"Overall positive rate: {preds.mean():.4f}")
print(f"Overall recall: {(preds[y==1].sum() / y.sum()):.4f}")

Overall positive rate: 0.1807
Overall recall: 0.8582


## 1. Demographic Parity (Flag Rate)

In [2]:
THRESHOLD_PP = 0.05  # 5 percentage points

parity_results = {}

for demo_col in ["gender", "race_ethnicity", "first_gen", "international_student", "financial_aid_status"]:
    rates = df.groupby(demo_col)["pred_label"].mean()
    spread = rates.max() - rates.min()
    flag = " ⚠️  > 5pp" if spread > THRESHOLD_PP else " ✓"
    parity_results[demo_col] = {"rates": rates, "spread": spread, "flag": flag}
    print(f"\n[{demo_col}] spread={spread:.4f}{flag}")
    for k, v in rates.sort_values(ascending=False).items():
        print(f"  {k}: {v:.4f}")


[gender] spread=0.0695 ⚠️  > 5pp
  male: 0.1846
  female: 0.1796
  non_binary: 0.1649
  other_not_listed: 0.1637
  prefer_not_to_say: 0.1152

[race_ethnicity] spread=0.0580 ⚠️  > 5pp
  native_american: 0.2265
  other: 0.1883
  pacific_islander: 0.1842
  hispanic_latino: 0.1838
  white: 0.1811
  black: 0.1790
  multiracial: 0.1745
  prefer_not_to_say: 0.1744
  asian: 0.1685

[first_gen] spread=0.0486 ✓
  1: 0.2119
  0: 0.1633

[international_student] spread=0.0009 ✓
  0: 0.1808
  1: 0.1798

[financial_aid_status] spread=0.1427 ⚠️  > 5pp
  both: 0.2485
  pell_grant: 0.2183
  loans_only: 0.1803
  none: 0.1059


## 2. Equalized Odds (True Positive Rate by Group)

In [3]:
# TPR = recall = P(predicted=1 | actual=1)
for demo_col in ["gender", "race_ethnicity", "first_gen"]:
    print(f"\nTPR (recall) by {demo_col}:")
    tpr_by_group = {}
    for group in df[demo_col].unique():
        mask = (df[demo_col] == group) & (y == 1)
        if mask.sum() < 10:  # skip groups with very few positives
            continue
        tpr = preds[mask].mean()
        tpr_by_group[group] = tpr
        print(f"  {group}: {tpr:.4f} (n={mask.sum():,})")
    
    if tpr_by_group:
        spread = max(tpr_by_group.values()) - min(tpr_by_group.values())
        flag = " ⚠️  > 5pp" if spread > THRESHOLD_PP else " ✓"
        print(f"  Spread: {spread:.4f}{flag}")


TPR (recall) by gender:
  male: 0.8680 (n=2,220)
  female: 0.8517 (n=2,974)
  non_binary: 0.8531 (n=143)
  other_not_listed: 0.8276 (n=29)
  prefer_not_to_say: 0.8182 (n=22)
  Spread: 0.0498 ✓

TPR (recall) by race_ethnicity:
  white: 0.8522 (n=2,700)
  pacific_islander: 0.9259 (n=27)
  hispanic_latino: 0.8815 (n=1,072)
  asian: 0.8525 (n=339)
  black: 0.8433 (n=721)
  native_american: 0.8824 (n=68)
  prefer_not_to_say: 0.8861 (n=79)
  multiracial: 0.8535 (n=273)
  other: 0.8532 (n=109)
  Spread: 0.0827 ⚠️  > 5pp

TPR (recall) by first_gen:
  1: 0.8715 (n=2,225)
  0: 0.8489 (n=3,163)
  Spread: 0.0226 ✓


## 3. False Positive Rate Parity

In [4]:
# FPR = P(predicted=1 | actual=0)
for demo_col in ["gender", "race_ethnicity", "first_gen"]:
    print(f"\nFPR by {demo_col}:")
    fpr_by_group = {}
    for group in df[demo_col].unique():
        mask = (df[demo_col] == group) & (y == 0)
        if mask.sum() < 10:
            continue
        fpr = preds[mask].mean()
        fpr_by_group[group] = fpr
        print(f"  {group}: {fpr:.4f} (n={mask.sum():,})")
    
    if fpr_by_group:
        spread = max(fpr_by_group.values()) - min(fpr_by_group.values())
        flag = " ⚠️  > 5pp" if spread > THRESHOLD_PP else " ✓"
        print(f"  Spread: {spread:.4f}{flag}")


FPR by gender:
  male: 0.0305 (n=9,842)
  female: 0.0344 (n=13,767)
  non_binary: 0.0279 (n=718)
  other_not_listed: 0.0282 (n=142)
  prefer_not_to_say: 0.0070 (n=143)
  Spread: 0.0274 ✓

FPR by race_ethnicity:
  white: 0.0330 (n=12,227)
  pacific_islander: 0.0240 (n=125)
  hispanic_latino: 0.0326 (n=4,946)
  asian: 0.0328 (n=1,708)
  black: 0.0317 (n=3,250)
  native_american: 0.0415 (n=241)
  prefer_not_to_say: 0.0142 (n=351)
  multiracial: 0.0270 (n=1,257)
  other: 0.0454 (n=507)
  Spread: 0.0311 ✓

FPR by first_gen:
  1: 0.0397 (n=8,519)
  0: 0.0286 (n=16,093)
  Spread: 0.0111 ✓


## 4. Fairlearn Metrics Dashboard

In [5]:
try:
    from fairlearn.metrics import MetricFrame, demographic_parity_difference, equalized_odds_difference
    from sklearn.metrics import recall_score, precision_score

    for demo_col in ["gender", "race_ethnicity", "first_gen"]:
        mf = MetricFrame(
            metrics={"recall": recall_score, "precision": precision_score},
            y_true=y,
            y_pred=preds,
            sensitive_features=df[demo_col],
        )
        dp_diff = demographic_parity_difference(y, preds, sensitive_features=df[demo_col])
        eo_diff = equalized_odds_difference(y, preds, sensitive_features=df[demo_col])
        print(f"\n[{demo_col}]")
        print(f"  Demographic parity difference: {dp_diff:.4f}")
        print(f"  Equalized odds difference: {eo_diff:.4f}")
        print(mf.by_groups.round(3))
except ImportError:
    print("fairlearn not installed — run 'uv sync' to install")

fairlearn not installed — run 'uv sync' to install


## 5. Mitigation — Threshold Adjustment (if disparity found)

If demographic parity difference > 5pp for any group, adjust threshold per group.
Document before/after. This cell runs the analysis; actual mitigation is only
applied if `apply_mitigation = True`.

In [6]:
apply_mitigation = False  # set True after reviewing results above

# Example: threshold adjustment for first_gen
print("Threshold adjustment analysis for first_gen:")
for group, label in [(0, "continuing-gen"), (1, "first-gen")]:
    mask = df["first_gen"] == group
    group_probs = probs[mask]
    group_y = y[mask]
    
    # Find threshold that equalizes FPR with overall FPR
    overall_fpr = preds[y == 0].mean()
    
    best_thresh = 0.5
    best_diff = 1.0
    for thresh in np.arange(0.3, 0.8, 0.01):
        group_preds = (group_probs >= thresh).astype(int)
        group_fpr = group_preds[group_y == 0].mean() if (group_y == 0).sum() > 0 else 0
        diff = abs(group_fpr - overall_fpr)
        if diff < best_diff:
            best_diff = diff
            best_thresh = thresh
    
    current_flag_rate = preds[mask].mean()
    adjusted_preds = (group_probs >= best_thresh).astype(int)
    adjusted_flag_rate = adjusted_preds.mean()
    print(f"  {label}: current threshold={bundle.threshold:.2f} flag_rate={current_flag_rate:.4f}")
    print(f"           adjusted threshold={best_thresh:.2f} flag_rate={adjusted_flag_rate:.4f}")

if not apply_mitigation:
    print("\nMitigation NOT applied. Set apply_mitigation=True after reviewing disparities.")

Threshold adjustment analysis for first_gen:
  continuing-gen: current threshold=0.50 flag_rate=0.1633
           adjusted threshold=0.45 flag_rate=0.1700
  first-gen: current threshold=0.50 flag_rate=0.2119
           adjusted threshold=0.60 flag_rate=0.1976

Mitigation NOT applied. Set apply_mitigation=True after reviewing disparities.


## 6. Bias Audit Summary

In [7]:
print("=" * 60)
print("BIAS AUDIT SUMMARY")
print("=" * 60)
print()

for demo_col, result in parity_results.items():
    print(f"  {demo_col}: spread={result['spread']:.4f}{result['flag']}")

print()
print("Notes for docs/model_cards/risk_classifier.md:")
print("  - Complete TPR and FPR tables above")
print("  - If any spread > 5pp: document before/after mitigation")
print("  - Cultural normality review: see notebook 04, cell 5")
print("  - Bias audit is a continuous commitment — re-run on any model retrain")

BIAS AUDIT SUMMARY

  gender: spread=0.0695 ⚠️  > 5pp
  race_ethnicity: spread=0.0580 ⚠️  > 5pp
  first_gen: spread=0.0486 ✓
  international_student: spread=0.0009 ✓
  financial_aid_status: spread=0.1427 ⚠️  > 5pp

Notes for docs/model_cards/risk_classifier.md:
  - Complete TPR and FPR tables above
  - If any spread > 5pp: document before/after mitigation
  - Cultural normality review: see notebook 04, cell 5
  - Bias audit is a continuous commitment — re-run on any model retrain
